Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn
!pip install pandas
!pip install tqdm
!pip install scikit-image
!pip install scipy
!pip install ace_tools
!pip install imbalanced-learn

Calling the Libraries:

In [ ]:
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import cv2
import glob
import numpy as np
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from skimage import exposure
from skimage.io import imread
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import cv2
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from collections import Counter
import glob
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import os

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs/001/L_Fore/01.bmp'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)


Train

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import normalize

# === CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (60, 120)
PATCH_SIZE = 10
stride = 10
NUM_IMAGES = 7  # Protocol 1: using 7 training images per finger
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
LBP_CONFIGS = [(1, 8), (1, 16), (2, 8)]
NUM_COMPONENTS = 47  # Number of 2DPCA components

# === RIU2 MAPPING ===
def get_riu2_mapping(P):
    table = np.zeros(2 ** P, dtype=np.uint8)
    for i in range(2 ** P):
        binary = [(i >> j) & 1 for j in range(P)]
        rotations = [binary[n:] + binary[:n] for n in range(P)]
        min_rotation = min(rotations)
        extended = min_rotation + [min_rotation[0]]
        transitions = sum(extended[j] != extended[j + 1] for j in range(P))
        if transitions <= 2:
            table[i] = sum(min_rotation)
        else:
            table[i] = P + 1
    return table

mapping_dict = {P: get_riu2_mapping(P) for _, P in LBP_CONFIGS}

# === LBP FEATURE EXTRACTOR ===
def extract_lbp_histogram(block, P, R, riu2_map):
    lbp = local_binary_pattern(block, P, R, method='ror').astype(np.uint16)
    lbp_mapped = riu2_map[lbp]
    hist, _ = np.histogram(lbp_mapped.ravel(), bins=np.arange(0, P + 3), density=True)
    return hist

# === FEATURE EXTRACTION (Strategy 2: fingers treated separately) ===
train_lbp_features = []
train_labels = []

print("\n🔄 Extracting RIU2-LBP features for each finger (Strategy 2)...")

subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for img_idx in range(1, NUM_IMAGES + 1):
        for finger in FINGER_LIST:
            fused_vector = []
            img_path = os.path.join(subject_path, finger, f"{img_idx:02d}.bmp")
            print(f"📥 Loading: {img_path}")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                print(f"❌ Missing: {img_path}")
                continue

            img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
            img = cv2.fastNlMeansDenoising(img, h=10)
            img = cv2.equalizeHist(img).astype(np.float64) / 255.0
            img = (img - np.mean(img)) / (np.std(img) + 1e-8)

            for y in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, stride):
                for x in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, stride):
                    block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                    block_hist = []
                    for R, P in LBP_CONFIGS:
                        hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                        block_hist.extend(hist)
                    fused_vector.extend(block_hist)

            if len(fused_vector) > 0:
                label = f"{subj}_{finger}_img{img_idx:02d}"
                train_lbp_features.append(fused_vector)
                train_labels.append(label)

# === NORMALIZE AND CONVERT TO ARRAY ===
train_lbp_features = np.array(train_lbp_features, dtype=np.float32)
train_lbp_features = normalize(train_lbp_features, norm='l2')
train_labels = np.array(train_labels)

print("\n✅ RIU2-LBP feature matrix shape:", train_lbp_features.shape)

# === 2DPCA ===
original_dim = train_lbp_features.shape[1]
height = int(np.sqrt(original_dim))
while original_dim % height != 0:
    height -= 1
width = original_dim // height
print(f"\n🔁 Reshaping each feature to: ({height}, {width})")
train_2d = [f.reshape(height, width) for f in train_lbp_features]

def compute_2dpca(images_2d, num_components):
    print("\n⚙️ Computing 2DPCA projection matrix...")
    n = len(images_2d)
    h, w = images_2d[0].shape
    mean_img = sum(images_2d) / n
    G_t = np.zeros((w, w))

    for i, img in enumerate(images_2d):
        A = img - mean_img
        G_t += A.T @ A
        if i < 3:
            print(f"  ➕ Sample {i+1} contribution added")

    G_t /= n
    eig_vals, eig_vecs = np.linalg.eigh(G_t)
    idx = np.argsort(-eig_vals)
    eig_vecs = eig_vecs[:, idx[:num_components]]
    print(f"✅ 2DPCA projection matrix shape: {eig_vecs.shape}")
    return eig_vecs

W = compute_2dpca(train_2d, NUM_COMPONENTS)

projected_features = []
for i, img in enumerate(train_2d):
    feat = img @ W
    projected_features.append(feat)
    if i < 3:
        print(f"🧮 Projected shape of sample {i+1}: {feat.shape}")

train_lbp_pca = np.array([f.flatten() for f in projected_features])
train_lbp_pca = normalize(train_lbp_pca, norm='l2')

print("\n✅ Final 2DPCA-transformed shape:", train_lbp_pca.shape)
print("✅ Number of principal components used:", W.shape[1])

Test

In [ ]:
# === TEST CONFIGURATION (Strategy 2) ===
TEST_INDICES = [8, 9, 10]
test_images = []
test_labels = []

for subj in tqdm(subject_dirs, desc="Loading Test Images"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for img_idx in TEST_INDICES:
        for finger in FINGER_LIST:
            fused_vector = []
            img_path = os.path.join(subject_path, finger, f"{img_idx:02d}.bmp")
            print(f"📥 Loading: {img_path}")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                print(f"❌ Missing: {img_path}")
                continue

            img = cv2.resize(img, (IMAGE_SIZE[1], IMAGE_SIZE[0]))
            img = cv2.fastNlMeansDenoising(img, h=10)
            img = cv2.equalizeHist(img).astype(np.float64) / 255.0
            img = (img - np.mean(img)) / (np.std(img) + 1e-8)

            for y in range(0, IMAGE_SIZE[0] - PATCH_SIZE + 1, stride):
                for x in range(0, IMAGE_SIZE[1] - PATCH_SIZE + 1, stride):
                    block = img[y:y + PATCH_SIZE, x:x + PATCH_SIZE]
                    block_hist = []
                    for R, P in LBP_CONFIGS:
                        hist = extract_lbp_histogram(block, P, R, mapping_dict[P])
                        block_hist.extend(hist)
                    fused_vector.extend(block_hist)

            if len(fused_vector) > 0:
                label = f"{subj}_{finger}_img{img_idx:02d}"
                test_images.append(fused_vector)
                test_labels.append(label)

# === RESHAPE & PROJECT TEST IMAGES INTO 2DPCA SPACE ===
test_images = np.array(test_images, dtype=np.float32)
test_images = normalize(test_images, norm='l2')
test_2d = [f.reshape(height, width) for f in test_images]

projected_test = []
for i, img in enumerate(test_2d):
    feat = img @ W
    projected_test.append(feat)
    if i < 3:
        print(f"🧪 Projected test image {i+1}: {feat.shape}")

test_flat_features = np.array([f.flatten() for f in projected_test])
test_flat_features = normalize(test_flat_features, norm='l2')
test_labels = np.array(test_labels)

print("✅ Test feature matrix shape:", test_flat_features.shape)


Benchmarking

In [ ]:
# === CLASSIFICATION FOR FINGER-LEVEL IDENTIFICATION ===
correct_matches = 0
total_tests = len(test_flat_features)

print("🔍 Starting classification using Manhattan distance (Match: Subject + Finger)...")

for i in range(total_tests):
    test_vec = test_flat_features[i]
    true_label = test_labels[i]  # Format: "subject_finger_imgXX"

    distances = np.sum(np.abs(train_lbp_pca - test_vec), axis=1)
    min_index = np.argmin(distances)
    predicted_label = train_labels[min_index]  # Format: "subject_finger_imgYY"

    # 🎯 Extract subject ID and finger
    true_id, true_finger = true_label.split('_')[0], true_label.split('_')[1]
    pred_id, pred_finger = predicted_label.split('_')[0], predicted_label.split('_')[1]

    if pred_id == true_id and pred_finger == true_finger:
        correct_matches += 1
        match_result = "✅"
    else:
        match_result = "❌"

    print(f"Test sample {i+1}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

accuracy = (correct_matches / total_tests) * 100
print(f"🎉 Finger-Level Identification Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests})")

Session Independent R5

In [ ]:
import numpy as np

# === CLASSIFICATION WITH (2D)²PCA Features — Rank-1 and Rank-5 ===
rank1_correct = 0
rank5_correct = 0
total_tests = len(test_flat_features)

print("\n🔍 Classification using Manhattan distance (2D²PCA - Match: Subject + Finger)...\n")

for i in range(total_tests):
    test_vec = test_flat_features[i]
    true_label = test_labels[i]  # Format: "subject_finger_imgXX"

    # 📏 Compute Manhattan distances to all training samples
    distances = np.sum(np.abs(train_lbp_pca - test_vec), axis=1)
    sorted_indices = np.argsort(distances)

    # === True identity (subject + finger)
    true_parts = true_label.split('_')
    true_id, true_finger = true_parts[0], true_parts[1]
    true_identity = f"{true_id}_{true_finger}"

    # === Rank-1 Match
    pred_label_r1 = train_labels[sorted_indices[0]]
    pred_parts_r1 = pred_label_r1.split('_')
    pred_identity_r1 = f"{pred_parts_r1[0]}_{pred_parts_r1[1]}"
    r1_match = (pred_identity_r1 == true_identity)
    if r1_match:
        rank1_correct += 1

    # === Rank-5 Match
    r5_match = False
    for k in range(5):
        pred_label = train_labels[sorted_indices[k]]
        pred_parts = pred_label.split('_')
        pred_identity = f"{pred_parts[0]}_{pred_parts[1]}"
        if pred_identity == true_identity:
            rank5_correct += 1
            r5_match = True
            break

    # Logging
    icon_r1 = "✅" if r1_match else "❌"
    icon_r5 = "✅" if r5_match else "❌"
    print(f"🔹 Test {i+1:04d}: True = {true_label}, Rank-1 Pred = {pred_label_r1} {icon_r1}, Rank-5 Match: {icon_r5}")

# 📊 Final results
rank1_accuracy = (rank1_correct / total_tests) * 100
rank5_accuracy = (rank5_correct / total_tests) * 100

print("\n📊 Final Results for (2D)²PCA")
print(f"🥇 Rank-1 Accuracy: {rank1_accuracy:.2f}% ({rank1_correct}/{total_tests})")
print(f"🖐️  Rank-5 Accuracy: {rank5_accuracy:.2f}% ({rank5_correct}/{total_tests})")


Session Independent CMC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ✅ Helper function to extract subject and finger from label
def extract_subject_and_finger(label):
    parts = label.split('_')
    if len(parts) < 3:
        raise ValueError(f"Label format must be 'subject_finger_imgXX', but got: {label}")
    return parts[0], parts[1]  # Assuming label = '000_f1_img08'

# === CONFIGURATION ===
max_rank = 100
rank_correct = np.zeros(max_rank)
total_tests = len(test_flat_features)

print("📊 Calculating Session-Independent CMC Curve (Matching: Subject + Finger)...")

for i in range(total_tests):
    proj_test = test_flat_features[i]
    true_label = test_labels[i]
    true_subj, true_finger = extract_subject_and_finger(true_label)
    true_id = f"{true_subj}_{true_finger}"

    # 🧮 Compute Manhattan distances to all training samples
    distances = np.sum(np.abs(train_lbp_pca - proj_test), axis=1)
    sorted_indices = np.argsort(distances)

    # 🔍 Find first correct match among sorted distances
    for r in range(max_rank):
        candidate_label = train_labels[sorted_indices[r]]
        cand_subj, cand_finger = extract_subject_and_finger(candidate_label)
        candidate_id = f"{cand_subj}_{cand_finger}"

        if candidate_id == true_id:
            rank_correct[r:] += 1
            break

# 📊 Normalize to percentages
cmc_curve = (rank_correct / total_tests) * 100

# 📈 Plotting the CMC curve
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, max_rank + 1), cmc_curve, label="Session-Independent CMC", linewidth=2)
plt.xlabel("Rank")
plt.ylabel("Identification Accuracy (%)")
plt.title("CMC Curve — Session-Independent (Subject + Finger Matching)")
plt.grid(True)
plt.xticks(np.arange(0, max_rank + 1, 10))
plt.xlim([1, max_rank])
plt.legend()
plt.tight_layout()
plt.show()

# 🧾 Print key rank accuracies
print("\n🎯 Key Rank Accuracies:")
print(f"🥇 Rank-1 Accuracy   : {cmc_curve[0]:.2f}%")
print(f"🖐️  Rank-5 Accuracy   : {cmc_curve[4]:.2f}%")
print(f"🔟 Rank-10 Accuracy  : {cmc_curve[9]:.2f}%")
print(f"💯 Rank-100 Accuracy : {cmc_curve[99]:.2f}%")


Session Independent Precision, Recall, F1, Accuracy

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# ✅ Helper function to extract subject and finger from a label
def extract_subject_and_finger(label):
    parts = label.split('_')
    if len(parts) < 3:
        raise ValueError(f"Invalid label format: {label}")
    return parts[0], parts[1]  # Assuming format: '000_f1_imgXX'

# === Initialize score and label containers
all_scores = []
all_labels = []

print("🔍 Starting Session-Independent Verification (Matching on Subject + Finger)...\n")

# === Pairwise comparisons
for test_idx in range(len(test_flat_features)):
    test_vec = test_flat_features[test_idx]
    test_label = test_labels[test_idx]
    test_subj, test_finger = extract_subject_and_finger(test_label)
    test_id = f"{test_subj}_{test_finger}"

    for train_idx in range(len(train_lbp_pca)):
        train_vec = train_lbp_pca[train_idx]
        train_label = train_labels[train_idx]
        train_subj, train_finger = extract_subject_and_finger(train_label)
        train_id = f"{train_subj}_{train_finger}"

        # Compute similarity (higher = more similar)
        score = -np.sum(np.abs(test_vec - train_vec))
        all_scores.append(score)

        # Label as genuine if subject + finger match
        is_genuine = int(test_id == train_id)
        all_labels.append(is_genuine)

# === Normalize scores to [0, 1]
scores = np.array(all_scores)
labels = np.array(all_labels)
if scores.max() != scores.min():
    scores = (scores - scores.min()) / (scores.max() - scores.min())
else:
    scores = np.zeros_like(scores)  # Avoid division by zero

# === Threshold search to maximize F1 score
best_f1 = best_thresh = best_prec = best_rec = 0

for t in np.linspace(0, 1, 1000):
    preds = (scores >= t).astype(int)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t
        best_prec = precision
        best_rec = recall

# === Final metrics at best threshold
final_preds = (scores >= best_thresh).astype(int)
accuracy = accuracy_score(labels, final_preds)

# === Output summary
print("📊 Verification Summary — Session-Independent (Subject + Finger)")
print(f"📍 Optimal Threshold  : {best_thresh:.3f}")
print(f"✔️ Accuracy           : {accuracy * 100:.2f}%")
print(f"✔️ Precision (PR)     : {best_prec * 100:.2f}%")
print(f"✔️ Recall (RC)        : {best_rec * 100:.2f}%")
print(f"✔️ F1 Score (F1)      : {best_f1 * 100:.2f}%")
